In [ ]:
# Cell 1 — Mount Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 2 — Paths
# BASE: the 'classification' folder on your Drive where everything lives
BASE        = '/content/drive/MyDrive/classification'
CLS_DATA    = f'{BASE}/Classification_TS'   # HAR/, Epilepsy-2/, eeg_no_big/ live here
RESULTS_CSV = f'{BASE}/results/layer_classification.csv'

In [ ]:
# Cell 3 — sys.path + working directory
import os, sys
os.chdir(BASE)
for p in [BASE,
          f'{BASE}/Discrete_JEPA',
          f'{BASE}/Discrete_JEPA/data_loaders']:
    if p not in sys.path:
        sys.path.insert(0, p)
print('CWD:', os.getcwd())

In [ ]:
# Cell 4 — Stage checkpoints to the paths each model expects
import shutil

LAYERS = [2, 4, 8, 12, 24]

def _cp(src, dst):
    if not os.path.exists(src):
        print(f'  MISSING  {src}'); return
    if os.path.exists(dst):
        print(f'  already  {os.path.relpath(dst, BASE)}'); return
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copy2(src, dst)
    print(f'  copied   {os.path.relpath(dst, BASE)}')

# DINO  →  checkpoints_layers{n}/checkpoint_best.pth
for n in LAYERS:
    _cp(f'{BASE}/Dino LAYER MONASH/dino_layers{n}.pth',
        f'{BASE}/checkpoints_layers{n}/checkpoint_best.pth')

# JEPA  →  output_model/JEPA_layers{n}/best_model.pt
for n in LAYERS:
    _cp(f'{BASE}/JEPA Layers monash/jepa_layers{n}.pt',
        f'{BASE}/output_model/JEPA_layers{n}/best_model.pt')

# LE-JEPA  →  output_model/LE-JEPA_layers{n}/best_model.pt
for n in LAYERS:
    _cp(f'{BASE}/LEJEPA LAYERS moansh/lejepa_layers{n}.pt',
        f'{BASE}/output_model/LE-JEPA_layers{n}/best_model.pt')

# PatchTST — build the expected filename from config
import importlib.util as _ilu

def _load_cfg(path, name):
    spec = _ilu.spec_from_file_location(name, path)
    mod  = _ilu.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod.config

ptst_cfg = _load_cfg(f'{BASE}/PatchTST_self_supervised/config_patchtst.py', 'config_patchtst')
ptst_dset   = ptst_cfg.get('pretrain_dataset', 'etth1')
model_type  = ptst_cfg.get('model_type', 'based_model')
ptst_fname  = (f"patchtst_pretrained_cw{ptst_cfg.get('context_points',512)}"
               f"_patch{ptst_cfg.get('patch_len',12)}_stride{ptst_cfg.get('stride',12)}"
               f"_epochs-pretrain{ptst_cfg.get('n_epochs_pretrain',10)}"
               f"_mask{ptst_cfg.get('mask_ratio',0.4)}_model{ptst_cfg.get('pretrained_model_id',1)}.pth")
for n in LAYERS:
    _cp(f'{BASE}/PatchTST - layers model/patchtst_layers{n}.pth',
        f'{BASE}/PatchTST_self_supervised/saved_models/{ptst_dset}/masked_patchtst/{model_type}/layers{n}/{ptst_fname}')

# NPT — build expected filename from config + _model_fname helper
sys.path.insert(0, f'{BASE}/NPT')
from ntp_pretrain import _model_fname as _npt_fname
npt_cfg   = _load_cfg(f'{BASE}/NPT/config_ntp.py', 'config_ntp')
npt_dset  = npt_cfg.get('pretrain_dataset', 'etth1')
npt_base  = _npt_fname(npt_cfg, npt_dset)
for n in LAYERS:
    _cp(f'{BASE}/NTP layers monash/npt_layers{n}.pt',
        f'{BASE}/NPT/saved_models/{npt_dset}/ntp/layers{n}/{npt_base}.pt')

print('\nCheckpoint staging complete.')

In [ ]:
# Cell 5 — Inject classification_data_dir into every model config file
# Each model loads its config fresh from disk, so we patch the files once.
import re

CONFIG_FILES = [
    f'{BASE}/TSDiNO/config.py',
    f'{BASE}/JEPA/config_files/config_jepa.py',
    f'{BASE}/LE-JEPA/config_lejepa.py',
    f'{BASE}/PatchTST_self_supervised/config_patchtst.py',
    f'{BASE}/NPT/config_ntp.py',
]

KEY   = 'classification_data_dir'
VALUE = CLS_DATA

for cfg_path in CONFIG_FILES:
    with open(cfg_path) as f:
        src = f.read()
    if KEY in src:
        # Already present — update the value
        src = re.sub(
            rf'("{KEY}"\s*:\s*)"[^"]*"',
            rf'\1"{VALUE}"',
            src
        )
        tag = 'updated'
    else:
        # Insert after the opening brace of config = {
        src = re.sub(
            r'(config\s*=\s*\{)',
            rf'\1\n    "{KEY}": "{VALUE}",',
            src,
            count=1
        )
        tag = 'injected'
    with open(cfg_path, 'w') as f:
        f.write(src)
    print(f'  {tag:8s}  {os.path.relpath(cfg_path, BASE)}')

print('\nAll configs patched.')

In [ ]:
# Cell 6 — Run classification sweep
import csv, traceback
from datetime import datetime
from pathlib import Path
from Train_and_downstream import run

MODELS   = ['dino', 'jepa_simple', 'lejepa', 'patchtst', 'npt']
LAYERS   = [2, 4, 8, 12, 24]
DATASETS = ['HAR', 'Epilepsy-2', 'eeg_no_big']

out_csv    = Path(RESULTS_CSV)
fieldnames = ['model', 'encoder_layers', 'dataset', 'accuracy', 'timestamp']
out_csv.parent.mkdir(parents=True, exist_ok=True)

# Resume support — skip already-completed combos
rows, done = [], set()
if out_csv.exists():
    with open(out_csv) as f:
        for row in csv.DictReader(f):
            rows.append(row)
            done.add((row['model'], int(row['encoder_layers']), row['dataset']))
    print(f'Resuming — {len(done)} combos already done.')

total = len(MODELS) * len(LAYERS) * len(DATASETS)
count = 0

for n_layers in LAYERS:
    for model in MODELS:
        for dataset in DATASETS:
            count += 1
            key = (model, n_layers, dataset)

            if key in done:
                print(f'[{count}/{total}] skip  {model} / layers{n_layers} / {dataset}')
                continue

            print(f'\n[{count}/{total}] {"="*50}')
            print(f'  model={model}  layers={n_layers}  dataset={dataset}')
            print(f'  {"="*50}')

            cls_acc = None
            try:
                result  = run(model=model, task='classify',
                              classification_dataset=dataset,
                              encoder_layers=n_layers)
                cls_acc = result[2] if isinstance(result, tuple) else result
                print(f'  => Accuracy: {cls_acc:.4f}')
            except Exception as e:
                print(f'  => ERROR: {e}')
                traceback.print_exc()

            rows.append({
                'model':          model,
                'encoder_layers': n_layers,
                'dataset':        dataset,
                'accuracy':       f'{cls_acc:.6f}' if cls_acc is not None else 'N/A',
                'timestamp':      datetime.now().isoformat(timespec='seconds'),
            })
            done.add(key)

            # Save after every run — survives Colab timeout
            with open(out_csv, 'w', newline='') as f:
                writer = csv.DictWriter(f, fieldnames=fieldnames)
                writer.writeheader()
                writer.writerows(rows)

print(f'\nDone. Results → {out_csv}')

In [ ]:
# Cell 7 — View results as a pivot table
import pandas as pd
df = pd.read_csv(RESULTS_CSV)
df['accuracy'] = pd.to_numeric(df['accuracy'], errors='coerce')

for dataset in df['dataset'].unique():
    print(f'\n=== {dataset} ===')
    pivot = (df[df['dataset'] == dataset]
             .pivot(index='model', columns='encoder_layers', values='accuracy')
             .round(4))
    print(pivot.to_string())